In [6]:
import pandas as pd

# Load datasets
hospital_overview = pd.read_excel("Final_Table_1_Hospital_Overview.xlsx")
admission = pd.read_csv("admission.csv")
ward = pd.read_csv("ward.csv")
readmission = pd.read_excel("readmission_cleaned.xlsx")

print("Files loaded successfully!")

Files loaded successfully!


In [7]:
print("Hospital Overview:")
print(hospital_overview.columns.tolist())

print("\nAdmission:")
print(admission.columns.tolist())

print("\nWard:")
print(ward.columns.tolist())

print("\nReadmission:")
print(readmission.columns.tolist())

Hospital Overview:
['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id', 'length_of_stay', 'gender', 'date_of_birth', 'blood_group', 'city', 'department_name', 'department_type', 'floor_number', 'status', 'ward_name', 'ward_type', 'total_beds', 'bed_number', 'bed_status', 'disease_name', 'disease_category', 'bill_id', 'bill_date', 'total_amount', 'insurance_covered_amount', 'patient_payable_amount', 'payment_status', 'payment_mode']

Admission:
['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id']

Ward:
['ward_id', 'ward_name', 'ward_type', 'total_beds', 'department_id']

Readmission:
['hospital_name', 'Admission_date', 'hospital_id', 'hospital_beds_available', 'occupied_beds', 'hospital_ward', 'patient_id', 'patient_gender', 'patient_age', 'patient_race', 'patient_sat_score', 'pati

In [8]:
# Total Admissions

total_admissions = admission["admission_id"].nunique()

print("Total Admissions:", total_admissions)

Total Admissions: 45000


In [9]:
print("Ward shape:", ward.shape)
print("Readmission shape:", readmission.shape)

print("\nWard total beds:")
print(ward["total_beds"].sum())

print("\nReadmission occupied beds:")
print(readmission["occupied_beds"].sum())

print("\nReadmission hospital beds available:")
print(readmission["hospital_beds_available"].sum())

Ward shape: (27, 5)
Readmission shape: (10000, 26)

Ward total beds:
415

Readmission occupied beds:
1990850

Readmission hospital beds available:
2991110


In [10]:
# Check occupied beds at hospital + date level

bed_check = readmission[
    ["hospital_name", "Admission_date", "hospital_beds_available", "occupied_beds"]
].drop_duplicates()

print("Original Readmission rows:", len(readmission))
print("Unique hospital-date-bed records:", len(bed_check))

print("\nSample:")
print(bed_check.head(10))

Original Readmission rows: 10000
Unique hospital-date-bed records: 9979

Sample:
                hospital_name Admission_date  hospital_beds_available  \
0  The Johns Hopkins Hospital      1/30/2024                      260   
1  The Johns Hopkins Hospital      3/20/2022                      250   
2  The Johns Hopkins Hospital      7/13/2021                      350   
3  The Johns Hopkins Hospital      1/26/2021                      190   
4  The Johns Hopkins Hospital       2/6/2024                      220   
5  The Johns Hopkins Hospital      4/29/2022                      420   
6  The Johns Hopkins Hospital      7/25/2020                      100   
7  The Johns Hopkins Hospital      5/18/2022                      350   
8  The Johns Hopkins Hospital      7/20/2022                      220   
9  The Johns Hopkins Hospital      5/29/2022                      450   

   occupied_beds  
0             90  
1             90  
2            220  
3            350  
4            390  
5

In [11]:
# Check unique hospitals and dates

print("Unique Hospitals:", bed_check["hospital_name"].nunique())
print("Unique Dates:", bed_check["Admission_date"].nunique())

print("\nHospital-wise records:")
print(bed_check.groupby("hospital_name").size())

Unique Hospitals: 1
Unique Dates: 1460

Hospital-wise records:
hospital_name
The Johns Hopkins Hospital    9979
dtype: int64


In [12]:
# Check daily bed information

daily_beds = (
    bed_check
    .groupby("Admission_date")[["hospital_beds_available", "occupied_beds"]]
    .mean()
    .reset_index()
)

print(daily_beds.head(10))

print("\nNumber of daily records:", len(daily_beds))

print("\nAverage available beds:", daily_beds["hospital_beds_available"].mean())
print("Average occupied beds:", daily_beds["occupied_beds"].mean())

  Admission_date  hospital_beds_available  occupied_beds
0       1/1/2021               323.333333     226.666667
1       1/1/2022               226.666667     103.333333
2       1/1/2023               290.000000     157.777778
3       1/1/2024               253.750000     160.000000
4      1/10/2021               328.333333     166.666667
5      1/10/2022               322.222222     160.000000
6      1/10/2023               231.666667     315.000000
7      1/10/2024               260.000000     245.000000
8      1/11/2021               311.111111     203.333333
9      1/11/2022               324.000000     138.000000

Number of daily records: 1460

Average available beds: 299.3723351705372
Average occupied beds: 199.12756976661086


In [13]:
# Check whether occupied beds exceed available beds

invalid_beds = daily_beds[
    daily_beds["occupied_beds"] > daily_beds["hospital_beds_available"]
]

print("Invalid daily records:", len(invalid_beds))

print("\nSample invalid records:")
print(invalid_beds.head(10))

Invalid daily records: 110

Sample invalid records:
    Admission_date  hospital_beds_available  occupied_beds
6        1/10/2023               231.666667     315.000000
11       1/11/2024               170.000000     208.333333
25       1/15/2022               178.333333     196.666667
46        1/2/2023               207.500000     302.500000
56       1/22/2021               260.000000     266.000000
58       1/22/2023               286.666667     300.000000
68       1/25/2021               177.500000     190.000000
115       1/7/2024               223.333333     263.333333
118       1/8/2023               283.333333     288.333333
138     10/12/2022               285.000000     302.500000


In [14]:
print("Maximum occupied beds:", readmission["occupied_beds"].max())
print("Minimum occupied beds:", readmission["occupied_beds"].min())

print("\nMaximum available beds:", readmission["hospital_beds_available"].max())
print("Minimum available beds:", readmission["hospital_beds_available"].min())

print("\nHMIS Total Beds:", ward["total_beds"].sum())

Maximum occupied beds: 400
Minimum occupied beds: 0

Maximum available beds: 500
Minimum available beds: 100

HMIS Total Beds: 415


In [15]:
# Occupancy Rate Calculation

average_occupied_beds = daily_beds["occupied_beds"].mean()
total_hospital_beds = ward["total_beds"].sum()

occupancy_rate = (average_occupied_beds / total_hospital_beds) * 100

print("===== OCCUPANCY RATE =====")
print("Average Daily Occupied Beds:", round(average_occupied_beds, 2))
print("Total Hospital Beds:", total_hospital_beds)
print("Occupancy Rate:", round(occupancy_rate, 2), "%")

===== OCCUPANCY RATE =====
Average Daily Occupied Beds: 199.13
Total Hospital Beds: 415
Occupancy Rate: 47.98 %


In [16]:
# ==============================
# FINAL KPI RESULTS - MEMBER 2
# ==============================

print("===== MEMBER 2 KPI RESULTS =====")

print("1. Total Admissions:", total_admissions)

print("2. Occupancy Rate:", round(occupancy_rate, 2), "%")

===== MEMBER 2 KPI RESULTS =====
1. Total Admissions: 45000
2. Occupancy Rate: 47.98 %


In [17]:
# Validate Total Admissions

admission_count = len(admission)
unique_admission_ids = admission["admission_id"].nunique()
hospital_overview_admissions = hospital_overview["admission_id"].nunique()

print("===== TOTAL ADMISSIONS VALIDATION =====")
print("Admission rows:", admission_count)
print("Unique Admission IDs:", unique_admission_ids)
print("Hospital Overview Unique Admission IDs:", hospital_overview_admissions)

if admission_count == unique_admission_ids == hospital_overview_admissions:
    print("Validation: PASSED")
else:
    print("Validation: FAILED")

===== TOTAL ADMISSIONS VALIDATION =====
Admission rows: 45000
Unique Admission IDs: 45000
Hospital Overview Unique Admission IDs: 45000
Validation: PASSED


In [18]:
total_admissions_df = pd.DataFrame({
    "KPI": ["Total Admissions"],
    "Formula": ["COUNT(DISTINCT admission_id)"],
    "Source Dataset": ["admission.csv"],
    "Source Field": ["admission_id"],
    "Result": [45000]
})

total_admissions_df.to_excel(
    "KPI_1_Total_Admissions.xlsx",
    index=False
)

print("KPI_1_Total_Admissions.xlsx created!")

KPI_1_Total_Admissions.xlsx created!


In [19]:
occupancy_rate_df = pd.DataFrame({
    "KPI": ["Occupancy Rate"],
    "Formula": ["(Average Daily Occupied Beds / Total Hospital Beds) × 100"],
    "Source Dataset": ["Readmission.csv + ward.csv"],
    "Source Fields": ["occupied_beds + total_beds"],
    "Average Daily Occupied Beds": [199.13],
    "Total Hospital Beds": [415],
    "Result": ["47.98%"]
})

occupancy_rate_df.to_excel(
    "KPI_2_Occupancy_Rate.xlsx",
    index=False
)

print("KPI_2_Occupancy_Rate.xlsx created!")

KPI_2_Occupancy_Rate.xlsx created!
